# 02a — Train MobileNet V3 Large

Binary coffee-bean classification (`defect` vs `non_defect`) on the Robusta
Lampung dataset. This notebook trains **only** `mobilenet_v3_large` using the
pre-computed deterministic split from `02_prepare_split.ipynb`.

**Prerequisites**: Run `02_prepare_split.ipynb` first to generate
`reports/split_indices.json`.

**Run on Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

Outputs:
- `checkpoints/best_model_mobilenet_v3_large.pth`
- `reports/history_mobilenet_v3_large.csv`
- `reports/curves_mobilenet_v3_large.png`
- `reports/timing_mobilenet_v3_large.json`

## 1 · Environment detection

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

Running in Colab: True


## 2 · Mount Drive & set paths

In [2]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah')
    WORK_ROOT  = Path('/content/work')   # fast local SSD on Colab
else:
    # Local fallback — repo root is parent of scripts/
    DRIVE_ROOT = Path('..').resolve()
    WORK_ROOT  = DRIVE_ROOT

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('WORK_ROOT  =', WORK_ROOT)

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah
WORK_ROOT  = /content/work


## 3 · Unzip / locate dataset

In [ ]:
import zipfile

DATA_ZIP  = DRIVE_ROOT / 'Dataset_Cropped.zip'
DATA_ROOT = WORK_ROOT

# Check if a representative subfolder of the dataset exists
if not (DATA_ROOT / 'defect').exists():
    if IN_COLAB:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(
                f'Dataset zip not found.\n'
                f'Please upload Dataset_Cropped.zip to:\n  {DATA_ZIP}'
            )
        print(f'Extracting {DATA_ZIP} -> {WORK_ROOT} ...')
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(WORK_ROOT)
        print('Done.')
    else:
        raise FileNotFoundError(
            f'Dataset folder not found at:\n  {DATA_ROOT}\n'
            f'Please run 01_auto_crop.py first to generate Dataset_Cropped/.'
        )
else:
    print(f'Dataset already present at {DATA_ROOT} — skipping extract.')

# Sanity check class folders
for c in ('defect', 'non_defect'):
    folder = DATA_ROOT / c
    if not folder.exists():
        raise FileNotFoundError(
            f'Expected class folder not found: {folder}\n'
            f'Dataset_Cropped/ must contain defect/ and non_defect/ subfolders.'
        )
    n = len(list(folder.glob('*.jpg')))
    print(f'  {c:>10}: {n:>5} images')

Extracting /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/Dataset_Cropped.zip -> /content/work ...
Done.
      defect:   500 images
  non_defect:   500 images


## 4 · Import utils.py

In [4]:
import importlib, shutil, sys

src_utils = DRIVE_ROOT / 'scripts' / 'utils.py'
if src_utils.exists():
    shutil.copy(src_utils, '/content/utils.py' if IN_COLAB else 'utils.py')
    sys.path.insert(0, '/content' if IN_COLAB else '.')
else:
    print(f'[warn] {src_utils} not found — paste utils.py manually.')

import utils  # noqa: E402
importlib.reload(utils)
from utils import (
    NUM_CLASSES, IMG_SIZE,
    seed_everything, get_transforms, TransformSubset,
    build_model, count_trainable_params, count_total_params,
)

## 5 · Hyperparameters & reproducibility

In [5]:
import torch

MODEL_NAME = "mobilenet_v3_large"

HPARAMS = {
    'batch_size'   : 64,   # T4 has 16 GB VRAM — 64 is safe for MobileNet
    'num_workers'  : 4,    # Colab has multi-core CPU; 4 workers saturates the GPU
    'epochs'       : 25,
    'learning_rate': 1e-4,
    'weight_decay' : 1e-4,
    'patience'     : 5,
    'split_ratios' : (0.8, 0.1, 0.1),
    'seed'         : 42,
    'img_size'     : 224,
}

seed_everything(HPARAMS['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True  # auto-tune convolutions for fixed 224x224 input
print('Device:', DEVICE, '| AMP:', USE_AMP)
if DEVICE.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
print('Model :', MODEL_NAME)
print('HPARAMS:', HPARAMS)

Device: cuda | AMP: True
GPU   : Tesla T4
Model : mobilenet_v3_large
HPARAMS: {'batch_size': 64, 'num_workers': 4, 'epochs': 25, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'patience': 5, 'split_ratios': (0.8, 0.1, 0.1), 'seed': 42, 'img_size': 224}


## 6 · Load & validate split indices

In [6]:
import json
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

# Load split_indices.json
split_path = DRIVE_ROOT / 'reports' / 'split_indices.json'
if not split_path.exists():
    raise FileNotFoundError("Run 02_prepare_split.ipynb first")

with open(split_path) as f:
    split = json.load(f)

# Load full dataset (no transform — TransformSubset handles it)
full_dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=None)
print('Classes:', full_dataset.class_to_idx)
print('Total images:', len(full_dataset))

# Validate basenames match
saved_basenames = [Path(p).name for p in split['samples']]
current_samples = [s for s, _ in full_dataset.samples]
current_basenames = [Path(p).name for p in current_samples]

if saved_basenames != current_basenames:
    raise ValueError(
        'Dataset basenames do not match split_indices.json.\n'
        'The dataset may have changed since the split was generated.\n'
        'Please re-run 02_prepare_split.ipynb to regenerate the split.'
    )
print('Basename validation: ✓ (all match)')

Classes: {'defect': 0, 'non_defect': 1}
Total images: 1000
Basename validation: ✓ (all match)


## 7 · Construct DataLoaders from split

In [7]:
# Reconstruct subsets via Subset (NOT random_split)
train_subset = Subset(full_dataset, split['train_indices'])
val_subset   = Subset(full_dataset, split['val_indices'])

# Wrap with transforms
train_ds = TransformSubset(train_subset, get_transforms(train=True))
val_ds   = TransformSubset(val_subset,   get_transforms(train=False))

common = dict(batch_size=HPARAMS['batch_size'],
              num_workers=HPARAMS['num_workers'],
              pin_memory=DEVICE.type == 'cuda')

train_loader = DataLoader(train_ds, shuffle=True,  drop_last=False, **common)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **common)

print(f'Train: {len(train_subset)} samples ({len(train_loader)} batches)')
print(f'Val  : {len(val_subset)} samples ({len(val_loader)} batches)')

Train: 800 samples (13 batches)
Val  : 100 samples (2 batches)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 8 · Training loop

In [8]:
import copy, time
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


def run_epoch(model, loader, criterion, optimizer, scaler, train: bool):
    """Run one epoch of training or validation."""
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x)
                loss   = criterion(logits, y)
            if train:
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            total    += x.size(0)
    return loss_sum / total, correct / total


# --- Build model ---
seed_everything(HPARAMS['seed'])
model     = build_model(MODEL_NAME).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(),
                  lr=HPARAMS['learning_rate'],
                  weight_decay=HPARAMS['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=HPARAMS['epochs'])
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(f'\n========== {MODEL_NAME} ==========')
print(f"Params: total={count_total_params(model):,} | "
      f"trainable={count_trainable_params(model):,}")

# --- Training ---
history: list = []
best_val_acc, best_epoch = -1.0, -1
best_state = None
epochs_no_improve = 0

t0 = time.time()
for epoch in range(1, HPARAMS['epochs'] + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion,
                                optimizer, scaler, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader, criterion,
                                optimizer, scaler, train=False)
    scheduler.step()

    lr = optimizer.param_groups[0]['lr']
    history.append({
        'epoch'     : epoch,
        'train_loss': tr_loss, 'train_acc': tr_acc,
        'val_loss'  : vl_loss, 'val_acc'  : vl_acc,
        'lr'        : lr,
    })

    improved = vl_acc > best_val_acc
    flag = '*' if improved else ' '
    elapsed = time.time() - t0
    avg_epoch_time = elapsed / epoch
    remaining = avg_epoch_time * (HPARAMS['epochs'] - epoch)
    elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed))
    eta_str     = time.strftime('%H:%M:%S', time.gmtime(remaining))

    print(f'  E{epoch:02d}{flag} | '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | '
          f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} | '
          f'lr={lr:.2e} | '
          f'elapsed={elapsed_str} ETA={eta_str}')

    if improved:
        best_val_acc, best_epoch = vl_acc, epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= HPARAMS['patience']:
            print(f'  Early stop at epoch {epoch} '
                  f"(no val_acc gain for {HPARAMS['patience']} epochs).")
            break

train_secs = time.time() - t0
total_m = int(train_secs // 60)
total_s = int(train_secs % 60)
print(f'  Training complete: best_epoch={best_epoch}, '
      f'best_val_acc={best_val_acc:.4f}, '
      f'total_time={total_m}m {total_s}s')

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 168MB/s]



========== mobilenet_v3_large ==========
Params: total=4,204,594 | trainable=4,204,594


/tmp/ipykernel_2443/489081457.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_2443/489081457.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  E01* | tr_loss=0.5891 tr_acc=0.7400 | vl_loss=0.5363 vl_acc=0.8200 | lr=9.96e-05 | elapsed=00:00:34 ETA=00:13:47
  E02* | tr_loss=0.3525 tr_acc=0.8662 | vl_loss=0.3792 vl_acc=0.8400 | lr=9.84e-05 | elapsed=00:00:40 ETA=00:07:40
  E03* | tr_loss=0.1977 tr_acc=0.9213 | vl_loss=0.2353 vl_acc=0.9300 | lr=9.65e-05 | elapsed=00:00:47 ETA=00:05:48
  E04* | tr_loss=0.1238 tr_acc=0.9525 | vl_loss=0.1912 vl_acc=0.9500 | lr=9.38e-05 | elapsed=00:00:52 ETA=00:04:37
  E05* | tr_loss=0.1171 tr_acc=0.9575 | vl_loss=0.1728 vl_acc=0.9600 | lr=9.05e-05 | elapsed=00:00:59 ETA=00:03:56
  E06  | tr_loss=0.1016 tr_acc=0.9575 | vl_loss=0.1527 vl_acc=0.9600 | lr=8.64e-05 | elapsed=00:01:05 ETA=00:03:27
  E07  | tr_loss=0.0753 tr_acc=0.9775 | vl_loss=0.1409 vl_acc=0.9600 | lr=8.19e-05 | elapsed=00:01:10 ETA=00:03:02
  E08* | tr_loss=0.0716 tr_acc=0.9675 | vl_loss=0.1422 vl_acc=0.9700 | lr=7.68e-05 | elapsed=00:01:18 ETA=00:02:46
  E09  | tr_loss=0.0686 tr_acc=0.9775 | vl_loss=0.2122 vl_acc=0.9100 | lr=7.13e-

## 9 · Save artefacts

In [9]:
import matplotlib.pyplot as plt

# --- Save checkpoint ---
ckpt_path = DRIVE_ROOT / 'checkpoints' / f'best_model_{MODEL_NAME}.pth'
torch.save({
    'model_name'  : MODEL_NAME,
    'state_dict'  : best_state,
    'best_epoch'  : best_epoch,
    'best_val_acc': best_val_acc,
    'hparams'     : HPARAMS,
    'class_to_idx': full_dataset.class_to_idx,
}, ckpt_path)
print(f'Saved checkpoint -> {ckpt_path}')

# --- Save history CSV ---
hist_df = pd.DataFrame(history)
csv_path = DRIVE_ROOT / 'reports' / f'history_{MODEL_NAME}.csv'
hist_df.to_csv(csv_path, index=False)
print(f'Saved history   -> {csv_path}')

# --- Save per-model curves PNG ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='val')
axes[0].set(title=f'{MODEL_NAME} — Loss', xlabel='epoch', ylabel='loss')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist_df['epoch'], hist_df['train_acc'], label='train')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'],   label='val')
axes[1].set(title=f'{MODEL_NAME} — Accuracy', xlabel='epoch', ylabel='accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()
png_path = DRIVE_ROOT / 'reports' / f'curves_{MODEL_NAME}.png'
fig.savefig(png_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved curves    -> {png_path}')

# --- Save timing JSON ---
timing_path = DRIVE_ROOT / 'reports' / f'timing_{MODEL_NAME}.json'
timing_payload = {
    'model_name'  : MODEL_NAME,
    'train_secs'  : round(train_secs, 2),
    'best_epoch'  : best_epoch,
    'best_val_acc': round(best_val_acc, 4),
}
timing_path.write_text(json.dumps(timing_payload, indent=2))
print(f'Saved timing    -> {timing_path}')

Saved checkpoint -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/checkpoints/best_model_mobilenet_v3_large.pth
Saved history   -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/history_mobilenet_v3_large.csv
Saved curves    -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/curves_mobilenet_v3_large.png
Saved timing    -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/timing_mobilenet_v3_large.json


## Done

Training of `mobilenet_v3_large` is complete. Artefacts saved:
- `checkpoints/best_model_mobilenet_v3_large.pth`
- `reports/history_mobilenet_v3_large.csv`
- `reports/curves_mobilenet_v3_large.png`
- `reports/timing_mobilenet_v3_large.json`

Next steps:
- Run `02b_train_resnet50.ipynb` and `02c_train_swin_tiny.ipynb` (any order)
- Then run `02d_compare_training.ipynb` to aggregate results